In [1]:
# --- headless execution harness (added by run_experiments.py) ---
import os as _h_os
import matplotlib as _h_mpl
_h_mpl.use('Agg')
import matplotlib.pyplot as _h_plt
_OUT = 'C:\\Users\\ASUS\\group_sgu26_deep_learning\\BTL03\\output\\1. MLP\\MNIST_MLP_pytorch'
_h_os.makedirs(_OUT, exist_ok=True)
_h_n = [0]
_h_show = _h_plt.show
def _saving_show(*args, **kwargs):
    _figs = [_h_plt.figure(n) for n in _h_plt.get_fignums()]
    for _f in _figs:
        _h_n[0] += 1
        _f.savefig(_h_os.path.join(_OUT, 'fig_%02d.png' % _h_n[0]), dpi=110, bbox_inches='tight')
    _h_show(*args, **kwargs)
    for _f in _figs:
        _h_plt.close(_f)
_h_plt.show = _saving_show
print('[harness] figures -> ' + _OUT)


[harness] figures -> C:\Users\ASUS\group_sgu26_deep_learning\BTL03\output\1. MLP\MNIST_MLP_pytorch


In [2]:
# Demo (PyTorch)

# Overview
- MNIST dataset
  + Goal: recognizing handwritten digits
  + Input: a 28x28 image
  + Output: class label (0, 1,..., 9)
- Defining an MLP for MNIST in PyTorch
- Training, evaluating, and extracting intermediate features (`fc2`)

# Import libraries

In [3]:
import time
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torchvision import datasets

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cuda


In [4]:
num_classes = 10   # 10 classes: 0, 1,..., 9

# (1) Data

In [5]:
# Download and load MNIST via torchvision
train_raw = datasets.MNIST(root='./data', train=True, download=True)
test_raw = datasets.MNIST(root='./data', train=False, download=True)

x_train = train_raw.data.numpy().reshape(60000, 28, 28).astype('float32') / 255.0
y_train = train_raw.targets.numpy().astype('int64')

x_test = test_raw.data.numpy().reshape(10000, 28, 28).astype('float32') / 255.0
y_test = test_raw.targets.numpy().astype('int64')

print(x_train.shape[0], 'train samples')
print(x_test.shape[0], 'test samples')

# Lưu ý: Trong PyTorch, nn.CrossEntropyLoss nhận trực tiếp nhãn số nguyên (0..9)
# nên không cần chuyển sang one-hot vector như keras.utils.to_categorical.

  0%|          | 0.00/9.91M [00:00<?, ?B/s]

  0%|          | 32.8k/9.91M [00:00<01:11, 138kB/s]

  1%|          | 98.3k/9.91M [00:00<00:45, 217kB/s]

  2%|▏         | 197k/9.91M [00:00<00:32, 301kB/s] 

  4%|▍         | 393k/9.91M [00:00<00:18, 502kB/s]

  8%|▊         | 819k/9.91M [00:01<00:09, 942kB/s]

... (20 outputs left out)

100%|██████████| 1.65M/1.65M [00:01<00:00, 1.12MB/s]

  0%|          | 0.00/4.54k [00:00<?, ?B/s]

100%|██████████| 4.54k/4.54k [00:00<00:00, 2.26MB/s]

60000 train samples
10000 test samples


# (2) Declare model

In [6]:
class MNIST_MLP(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28 * 28, 512)
        self.dropout1 = nn.Dropout(p=0.2)
        self.fc2 = nn.Linear(512, 512)
        self.dropout2 = nn.Dropout(p=0.2)
        self.fc3_10ways = nn.Linear(512, num_classes)

    def forward(self, x, return_fc2=False):
        x = self.flatten(x)
        x = self.dropout1(F.relu(self.fc1(x)))
        fc2_out = F.relu(self.fc2(x))
        x = self.dropout2(fc2_out)
        logits = self.fc3_10ways(x)
        if return_fc2:
            return fc2_out, logits
        return logits

model = MNIST_MLP(num_classes=num_classes).to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total_params:,} | Trainable params: {trainable_params:,}')

MNIST_MLP(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=784, out_features=512, bias=True)
  (dropout1): Dropout(p=0.2, inplace=False)
  (fc2): Linear(in_features=512, out_features=512, bias=True)
  (dropout2): Dropout(p=0.2, inplace=False)
  (fc3_10ways): Linear(in_features=512, out_features=10, bias=True)
)
Total params: 669,706 | Trainable params: 669,706


In [7]:
# Hiển thị bảng kiến trúc chi tiết (tương đương model.summary / plot_model)
try:
    from torchinfo import summary
    print(summary(model, input_size=(1, 28, 28), device=device))
except ImportError:
    print('Cài đặt torchinfo nếu muốn xem bảng summary chi tiết: pip install torchinfo')

Layer (type:depth-idx)                   Output Shape              Param #
MNIST_MLP                                [1, 10]                   --
├─Flatten: 1-1                           [1, 784]                  --
├─Linear: 1-2                            [1, 512]                  401,920
├─Dropout: 1-3                           [1, 512]                  --
├─Linear: 1-4                            [1, 512]                  262,656
├─Dropout: 1-5                           [1, 512]                  --
├─Linear: 1-6                            [1, 10]                   5,130
Total params: 669,706
Trainable params: 669,706
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.67
Input size (MB): 0.00
Forward/backward pass size (MB): 0.01
Params size (MB): 2.68
Estimated Total Size (MB): 2.69


# (3) Train defined model
- Training history (`loss`, `accuracy`, `val_loss`, `val_accuracy`) được lưu trong biến `history`
- Nếu gặp lỗi tràn VRAM (OOM), hãy giảm `batch_size`

In [8]:
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for bx, by in loader:
            bx, by = bx.to(device), by.to(device)
            logits = model(bx)
            loss = criterion(logits, by)
            total_loss += loss.item() * bx.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == by).sum().item()
            total += bx.size(0)
    return total_loss / total, correct / total


# declare learning rate, loss function, optimizer
lr = 0.01
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=lr)

batch_size = 128
epochs = 30

train_loader = DataLoader(
    TensorDataset(torch.from_numpy(x_train), torch.from_numpy(y_train)),
    batch_size=batch_size, shuffle=True
)
test_loader = DataLoader(
    TensorDataset(torch.from_numpy(x_test), torch.from_numpy(y_test)),
    batch_size=batch_size, shuffle=False
)

history = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}

starting_time = time.time()
for epoch in range(1, epochs + 1):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        logits = model(bx)
        loss = criterion(logits, by)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * bx.size(0)
        correct += (logits.argmax(dim=1) == by).sum().item()
        total += bx.size(0)

    train_loss = running_loss / total
    train_acc = correct / total
    val_loss, val_acc = evaluate(model, test_loader, criterion, device)

    history['loss'].append(train_loss)
    history['accuracy'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_accuracy'].append(val_acc)

    print(f'Epoch {epoch:2d}/{epochs} - loss: {train_loss:.4f} - accuracy: {train_acc:.4f} - val_loss: {val_loss:.4f} - val_accuracy: {val_acc:.4f}')

print('> training time is %.4f minutes' % ((time.time() - starting_time) / 60))

Epoch  1/30 - loss: 2.1226 - accuracy: 0.4637 - val_loss: 1.6933 - val_accuracy: 0.7239


Epoch  2/30 - loss: 1.1194 - accuracy: 0.7514 - val_loss: 0.6801 - val_accuracy: 0.8319


Epoch  3/30 - loss: 0.6179 - accuracy: 0.8297 - val_loss: 0.4715 - val_accuracy: 0.8747


Epoch  4/30 - loss: 0.4915 - accuracy: 0.8612 - val_loss: 0.3969 - val_accuracy: 0.8913


Epoch  5/30 - loss: 0.4317 - accuracy: 0.8758 - val_loss: 0.3593 - val_accuracy: 0.8984


Epoch  6/30 - loss: 0.3970 - accuracy: 0.8857 - val_loss: 0.3346 - val_accuracy: 0.9042


... (20 outputs left out)

Epoch 27/30 - loss: 0.1800 - accuracy: 0.9482 - val_loss: 0.1576 - val_accuracy: 0.9529


Epoch 28/30 - loss: 0.1735 - accuracy: 0.9497 - val_loss: 0.1531 - val_accuracy: 0.9550


Epoch 29/30 - loss: 0.1684 - accuracy: 0.9510 - val_loss: 0.1485 - val_accuracy: 0.9565


Epoch 30/30 - loss: 0.1645 - accuracy: 0.9523 - val_loss: 0.1462 - val_accuracy: 0.9568
> training time is 2.0952 minutes


# (4) Evaluate trained model

In [9]:
test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print('Test loss:', test_loss)
print('Test accuracy:', test_acc)

Test loss: 0.1462134002417326
Test accuracy: 0.9568


In [10]:
# select a test image randomly
random_test_index = np.random.choice(x_test.shape[0], size=1)[0]
test_img = x_test[random_test_index]
test_label = int(y_test[random_test_index])

# predict test image with trained model
model.eval()
with torch.no_grad():
    inp = torch.from_numpy(np.expand_dims(test_img, axis=0)).to(device)
    probs = F.softmax(model(inp), dim=1).cpu().numpy()
pred_label = int(np.argmax(probs))

plt.imshow(test_img, cmap='gray')
plt.title('true label = %d, predicted label = %d' % (test_label, pred_label))
plt.show()

C:\Users\ASUS\AppData\Local\Temp\ipykernel_10816\4017412204.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  _h_show(*args, **kwargs)


In [11]:
# Kiểm tra chi tiết kích thước & xác suất dự đoán (tương đương các cell cuối bản Keras)
random_test_index = np.random.choice(x_test.shape[0], size=1)[0]
test_img = x_test[random_test_index]
test_label = int(y_test[random_test_index])

plt.imshow(test_img, cmap='gray')
plt.show()
print('test_label:', test_label)
print('test_img.shape:', test_img.shape)
print('expanded shape:', np.expand_dims(test_img, axis=0).shape)

model.eval()
with torch.no_grad():
    inp = torch.from_numpy(np.expand_dims(test_img, axis=0)).to(device)
    y_pred = F.softmax(model(inp), dim=1).cpu().numpy()

print('y_pred.shape:', y_pred.shape)
print('sum(y_pred):', np.sum(y_pred, axis=1))
print('argmax(y_pred):', np.argmax(y_pred, axis=1))

test_label: 9
test_img.shape: (28, 28)
expanded shape: (1, 28, 28)
y_pred.shape: (1, 10)
sum(y_pred): [0.99999994]
argmax(y_pred): [4]


C:\Users\ASUS\AppData\Local\Temp\ipykernel_10816\4017412204.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  _h_show(*args, **kwargs)


### Trích xuất đặc trưng tầng ẩn `fc2` (tương đương `Model(inputs=model.input, outputs=[model.get_layer('fc2').output, model.output])`)

In [12]:
model.eval()
with torch.no_grad():
    inp = torch.from_numpy(np.expand_dims(test_img, axis=0)).to(device)
    fc2_feat, logits = model(inp, return_fc2=True)
    y_feat = fc2_feat.cpu().numpy()
    y_pred = F.softmax(logits, dim=1).cpu().numpy()

print('y_feat.shape, y_pred.shape:', y_feat.shape, y_pred.shape)
print('First 20 values of y_feat:', y_feat[0, :20])

y_feat.shape, y_pred.shape: (1, 512) (1, 10)
First 20 values of y_feat: [0.         0.24449065 0.50684494 0.         0.13154268 0.47009206
 0.         0.50131905 0.         0.59649223 0.36006162 0.
 0.00215076 0.08053982 0.         0.19458042 0.56762916 0.4135446
 0.         0.25503054]
